**Data sets per year for age group 16-29 years old (including 16 and 29)**

In [1]:
import pyreadstat
import polars as pl
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt

In [2]:
## Create a new dataframe "YYYY_AG1629" from ...pers... and fill with filtered list, filter set to alterx=16-29

## for 2001 - 2022, the age groups are:
#   ...
#   7 = 7 bis 9 Jahre
#   10-74 = Jahre
#   75 = 75 bis 79 Jahre
#   ...

## for 1991 the age groups are:
# ...
# 4 = 12-15
# 5 = 15-20
# 6 = 20-25
# 7 = 25-30
# 8 = 30-35
# ...

## 1991
df_1991_pandas, metadata = pyreadstat.read_por("1991/zbe_ps.por")
df_1991 = pl.from_pandas(df_1991_pandas)
df_1991_AG1629 = df_1991.filter(
    pl.col('E51X').is_between(5,7))

## 2001
df_2001 = pl.read_csv('2001/zbe01_02puf_p.csv', separator=",", ignore_errors=True)
df_2001_AG1629 = df_2001.filter(
    pl.col('PH01B2X').is_between(16,29))

## 2012
df_2012 = pl.read_csv('2012/zve13_puf_pers.csv', separator=";", ignore_errors=True)
df_2012_AG1629 = df_2012.filter(
    pl.col('alterx').is_between(16,29))

## 2022
df_2022 = pl.read_csv('2022/zve_pers_puf_2022_v1.csv', separator=";", ignore_errors=True)
df_2022_AG1629 = df_2022.filter(
    pl.col('alterx').is_between(16,29))

In [3]:
## Create a new dataframe "YYYY_AG16-29_activites" from ...sum... with only rows according to listed id_persx of "YYYY_AG16-29"

## 1991
pers_ids_1991_AG1629 = df_1991_AG1629.get_column('ID1').unique()

df_1991_activities_pandas, metadata = pyreadstat.read_por("1991/zbe_sum.por")
df_1991_activities = pl.from_pandas(df_1991_activities_pandas)

df_1991_AG1629_activities = df_1991_activities.filter(
    pl.col('ID1').is_in(pers_ids_1991_AG1629.implode()))

## 2001
pers_ids_2001_AG1629 = df_2001_AG1629.get_column('IDPERS').unique()

df_2001_activities = pl.read_csv('2001/zbe01_02puf_sum.csv', separator=",", ignore_errors=True)

df_2001_AG1629_activities = df_2001_activities.filter(
    pl.col('IDPERS').is_in(pers_ids_2001_AG1629.implode()))

## 2012
pers_ids_2012_AG1629 = df_2012_AG1629.get_column('id_persx').unique()

df_2012_activities = pl.read_csv('2012/zve13_puf_sum.csv', separator=";", ignore_errors=True)

df_2012_AG1629_activities = df_2012_activities.filter(
    pl.col('id_persx').is_in(pers_ids_2012_AG1629.implode()))

## 2022
pers_ids_2022_AG1629 = df_2022_AG1629.get_column('id_persx').unique()

df_2022_activities = pl.read_csv('2022/zve_sum_puf_2022_v1.csv', separator=";", ignore_errors=True)

df_2022_AG1629_activities = df_2022_activities.filter(
    pl.col('id_persx').is_in(pers_ids_2022_AG1629.implode()))

In [4]:
## Trim the dataframes to only the activity columns
df_1991_AG1629_activities = df_1991_AG1629_activities.select(
    pl.col(r'^ZH.*$')
).select(
    pl.exclude('ZHOEFF','ZHFAM','ZHPERS')
)

df_2001_AG1629_activities = df_2001_AG1629_activities.select(
    pl.col(r'^ZH.*$')
)

df_2012_AG1629_activities = df_2012_AG1629_activities.select(
    pl.col(r'^ha_.*$')
)

df_2022_AG1629_activities = df_2022_AG1629_activities.select(
    pl.col(r'^ha_.*$')
)

In [5]:
## Harmonise the time values to minutes

# 1991 uses minutes (f64), 2001 uses minutes (i64), 2012 uses seconds (i64), 2022 uses hh:mm (str)

# 1991
# already in minutes

# 2001
# already in minutes

# 2012 
# divide all values by 60 to go to minutes
df_2012_AG1629_activities = df_2012_AG1629_activities.with_columns(
        pl.all() / 60
    )

# 2022
# use left two digits of ":" and multiply by 60 and right two digits to add onto that value to go to minutes
for col in df_2022_AG1629_activities.columns:
    duration_in_minutes = []

    for duration_as_str in df_2022_AG1629_activities[col]:
        hours, minutes = duration_as_str.split(':')
        duration_in_minutes.append(int(hours) * 60 + int(minutes))

    df_2022_AG1629_activities = df_2022_AG1629_activities.with_columns(
        pl.Series(col, duration_in_minutes)
    )

In [6]:
## Create a new dataframe "df_YYYY_AG1629_activites_means" with the activity codes as rows and a column for YYYY
## for each column in df_activities: create the mean and write to the row of the activity code and the column YYYY

# Create a nested list with pairs of column and mean of column: [[ha_1, mean(ha_1)], [...]]
def column_and_mean(dataframe):
    
    rows = []
    
    for column in dataframe.columns:
        column_mean = dataframe.get_column(column).mean()
        rows.append([column, column_mean])
    
    return rows

## 1991
df_1991_AG1629_activities_means = pl.DataFrame(
    column_and_mean(df_1991_AG1629_activities),
    schema = ['activity code', 'mean'],
    orient = 'row'
)

## 2001
df_2001_AG1629_activities_means = pl.DataFrame(
    column_and_mean(df_2001_AG1629_activities),
    schema = ['activity code', 'mean'],
    orient = 'row'
)

## 2012
df_2012_AG1629_activities_means = pl.DataFrame(
    column_and_mean(df_2012_AG1629_activities),
    schema = ['activity code', 'mean'],
    orient = 'row'
)

## 2022
df_2022_AG1629_activities_means = pl.DataFrame(
    column_and_mean(df_2022_AG1629_activities),
    schema = ['activity code', 'mean'],
    orient = 'row'
)

In [7]:
df_codes_2012 = pl.DataFrame(
    df_2012_AG1629_activities_means['activity code'],
    schema=['codes'],
    orient= 'row'
)

df_codes_2012.write_csv('2001/codes_2012.csv', separator=',')

In [8]:
## New codes for 1991 -> 2001.
df_mapping_1991_to_2001 = pl.read_csv(
    "1991/codes_1991_to_2001.csv",
    separator=','
)

## New codes for 2001 -> 2012.
df_mapping_2001_to_2012 = pl.read_csv(
    "2001/codes_2001_to_2012.csv", 
    separator=','
)

In [9]:
## Map codes

def map_codes(df_map, df_data):
    code = df_data.columns[0]
    value = df_data.columns[1]

    mapping = df_map.select(
        pl.col(df_map.columns[0]).alias(code),
        pl.col(df_map.columns[1])
        .str.split("/")
        .alias("_mapped_codes"),
    )

    joined = df_data.join(mapping, on=code, how="left")

    # Keep codes that have no mapping as a one item list
    joined = joined.with_columns(
        pl.when(pl.col("_mapped_codes").is_null())
        .then(pl.concat_list(pl.col(code)))
        .otherwise(pl.col("_mapped_codes"))
        .alias("_mapped_codes")
    )

    # Count mapped codes so the value can be divided evenly
    joined = joined.with_columns(
        pl.col("_mapped_codes").list.len().alias("_split_count")
    )

    # Create one row per mapped code and divide the value
    result = (
        joined
        .explode("_mapped_codes")
        .with_columns(
            pl.col("_mapped_codes").alias(code),
            (pl.col(value) / pl.col("_split_count")).alias(value),
        )
        .drop("_mapped_codes", "_split_count")
    )

    return result

## Map 1991 to 2001
df_1991_AG1629_activities_means = map_codes(
    df_mapping_1991_to_2001, 
    df_1991_AG1629_activities_means
    )

## Map 1991 to 2012
df_1991_AG1629_activities_means = map_codes(
    df_mapping_2001_to_2012, 
    df_1991_AG1629_activities_means
    )

## Map 2001 to 2012
df_2001_AG1629_activities_means = map_codes(
    df_mapping_2001_to_2012, 
    df_2001_AG1629_activities_means
    )

## Rename ZH... activity codes to ha_...
def rename_activity_code(value):
    if value.startswith("ZH"):
        return "ha_" + str(value[2:])
    return value

df_1991_AG1629_activities_means = df_1991_AG1629_activities_means.with_columns(
    pl.col('activity code')
    .map_elements(rename_activity_code, return_dtype=pl.String)
    .alias('activity code')
)

df_2001_AG1629_activities_means = df_2001_AG1629_activities_means.with_columns(
    pl.col('activity code')
    .map_elements(rename_activity_code, return_dtype=pl.String)
    .alias('activity code')
)

## merging rows with the same code, sum of means
df_1991_AG1629_activities_means = df_1991_AG1629_activities_means.group_by('activity code').agg(pl.col('mean').sum())
df_2001_AG1629_activities_means = df_2001_AG1629_activities_means.group_by('activity code').agg(pl.col('mean').sum())

/var/folders/5_/0wmfbw9s5y34jq82848gv2qw0000gn/T/ipykernel_92308/3228498498.py:32: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("_mapped_codes")


**Data set for yearly comparison**

In [10]:
## Create a new dataframe "df_Year-by-Year_AG1629_overview" with the activity codes as rows and 1991, 2001, 2012 and 2022 as columns
## Add all data points of the years from dataframes "df_YYYY_AG1629_activites" to "df_Year-by-Year_AG1629_overview"

## Create rows with the activity code, find the row in 1991, 2001, 2012 and 2022 and add their means to the row. Return the rows.

tables = [
    df_1991_AG1629_activities_means.rename({"mean": "1991"}),
    df_2001_AG1629_activities_means.rename({"mean": "2001"}),
    df_2012_AG1629_activities_means.rename({"mean": "2012"}),
    df_2022_AG1629_activities_means.rename({"mean": "2022"}),
]

df_YBY_AG1629_means = tables[0]

for table in tables[1:]:
    df_YBY_AG1629_means = df_YBY_AG1629_means.join(
        table,
        on="activity code",
        how="full",
        coalesce=True,
    )

In [11]:
## Fill "null" with 0.0
df_YBY_AG1629_means = df_YBY_AG1629_means.with_columns(
    pl.col('1991').fill_null(0.0),
    pl.col('2001').fill_null(0.0),
    pl.col('2012').fill_null(0.0),
    pl.col('2022').fill_null(0.0),
)

## Add trends to compare year-over-year and a trend for 1991-2022 for each activity code for "df_Year-by-Year_AG1629_overview"

df_YBY_AG1629_means = df_YBY_AG1629_means.with_columns(
    (pl.col('2001') - pl.col('1991')).alias('1991 -> 2001')
)

df_YBY_AG1629_means = df_YBY_AG1629_means.with_columns(
    (pl.col('2012') - pl.col('2001')).alias('2001 -> 2012')
)

df_YBY_AG1629_means = df_YBY_AG1629_means.with_columns(
    (pl.col('2022') - pl.col('2012')).alias('2012 -> 2022')
)

df_YBY_AG1629_means = df_YBY_AG1629_means.with_columns(
    (pl.col('2022') - pl.col('1991')).alias('1991 -> 2022')
)

In [ ]:
## Get rid of wrong sums for the code groups
df_YBY_AG1629_means = df_YBY_AG1629_means.filter(
    pl.col('activity code').str.len_chars() > 5
)

activity code,1991,2001,2012,2022,1991 -> 2001,2001 -> 2012,2012 -> 2022,1991 -> 2022
str,f64,f64,f64,f64,f64,f64,f64,f64
"""ha_354""",44.610354,9.720763,0.0,0.0,-34.889591,-9.720763,0.0,-44.610354
"""ha_369""",35.207437,1.580347,0.959493,0.373519,-33.62709,-0.620854,-0.585974,-34.833918
"""ha_641""",36.837339,8.948541,10.0611,19.29933,-27.888798,1.112559,9.23823,-17.538008
"""ha_611""",23.403874,31.378565,35.286264,7.546368,7.974692,3.907699,-27.739896,-15.857506
"""ha_961""",15.355093,11.713317,2.604662,1.264812,-3.641776,-9.108655,-1.33985,-14.090281
…,…,…,…,…,…,…,…,…
"""ha_120""",68.622669,100.136627,86.042091,87.154817,31.513958,-14.094536,1.112726,18.532149
"""ha_614""",0.0,0.0,0.0,19.680577,0.0,0.0,19.680577,19.680577
"""ha_353""",0.0,0.0,17.384024,26.737506,0.0,17.384024,9.353483,26.737506


In [34]:
## Add label to activity codes
df_labels = pl.read_excel('2022/zve_2022_puf_dsb_v2_0.xlsx', engine='calamine', sheet_name='Summendaten')

df_labels = df_labels.drop(['Teil', 'Schlüssel'])

df_labels = df_labels.filter(
    pl.col('Variable').str.starts_with('ha_')
)

def map_labels(df_labels, df_data):
    map = dict(df_labels.iter_rows())
    activity_code = df_data.columns[0]
    return df_data.with_columns(
        pl.col(activity_code)
        .replace(map)
        .alias('label')
    )

df_YBY_AG1629_means = map_labels(
    df_labels, 
    df_YBY_AG1629_means
    )

## Clean up the data set to display the columns in the correct way and sort by the trend 1991 -> 2022
df_YBY_AG1629_means = df_YBY_AG1629_means.select('activity code', 'label', '1991', '2001', '2012', '2022', '1991 -> 2001', '2001 -> 2012', '2012 -> 2022', '1991 -> 2022').sort('1991 -> 2022')

df_YBY_AG1629_means

/var/folders/5_/0wmfbw9s5y34jq82848gv2qw0000gn/T/ipykernel_92308/1676534843.py:2: FutureWarning: from_arrow(<ArrowStreamExportable>) will return a Series instead of a DataFrame in 2.0. To avoid this warning, pass the ArrowStreamExportable to either `pl.DataFrame` or `pl.Series` instead based on your desired output type.
  df_labels = pl.read_excel('2022/zve_2022_puf_dsb_v2_0.xlsx', engine='calamine', sheet_name='Summendaten')


activity code,label,1991,2001,2012,2022,1991 -> 2001,2001 -> 2012,2012 -> 2022,1991 -> 2022
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""ha_354""","""ha_354""",44.610354,9.720763,0.0,0.0,-34.889591,-9.720763,0.0,-44.610354
"""ha_369""","""369 Sonstige/unbestimmte Aktiv…",35.207437,1.580347,0.959493,0.373519,-33.62709,-0.620854,-0.585974,-34.833918
"""ha_641""","""641 Zu Besuch sein/Besuch empf…",36.837339,8.948541,10.0611,19.29933,-27.888798,1.112559,9.23823,-17.538008
"""ha_611""","""611 Kontakte/Gespräche mit Hau…",23.403874,31.378565,35.286264,7.546368,7.974692,3.907699,-27.739896,-15.857506
"""ha_961""","""961 Wegezeiten Soziale Kontakt…",15.355093,11.713317,2.604662,1.264812,-3.641776,-9.108655,-1.33985,-14.090281
…,…,…,…,…,…,…,…,…,…
"""ha_120""","""120 Essen und Trinken (HA)""",68.622669,100.136627,86.042091,87.154817,31.513958,-14.094536,1.112726,18.532149
"""ha_614""","""614 Nutzung sozialer Medien (H…",0.0,0.0,0.0,19.680577,0.0,0.0,19.680577,19.680577
"""ha_353""","""353 Vor-/Nachbereitung Lehrver…",0.0,0.0,17.384024,26.737506,0.0,17.384024,9.353483,26.737506


In [35]:
## Write dataframe "df_Year-by-Year_AG1629_overview" to "Year-by-Year_AG1629_overview.csv"
df_YBY_AG1629_means.write_csv('YBY_AG1629_overview.csv', separator=',')